In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torchaudio
from torchaudio.functional import add_noise
from torch.utils.data import Dataset, DataLoader, random_split
from torchaudio.transforms import Spectrogram  # 수정: nnAudio Gammatonegram -> STFT Spectrogram
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SR = 8000
N_FFT = 256             # 수정: 512 -> 256
N_BINS = N_FFT // 2 + 1
HOP_LENGTH = 128
DURATION_SEC = 2
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SirenGRU(nn.Module):
    def __init__(self, input_size=N_BINS, hidden_size=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False  # 실시간 처리라 단방향
        )
        self.fc = nn.Linear(hidden_size, input_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x) # _ : 맨 마지막 순간의 내부 기억 상태 -> 필요 x
        return self.sigmoid(self.fc(out))

In [ ]:
class SirenClassificationDataset(Dataset):
    def __init__(self, clean_paths, noise_paths):
        self.clean_paths = clean_paths
        self.noise_paths = noise_paths
        self.max_len = SR * DURATION_SEC

        self.stft = Spectrogram(
            n_fft=N_FFT, hop_length=HOP_LENGTH, power=1.0
        ).to(DEVICE)

        self.filter_model = SirenGRU(input_size=N_BINS).to(DEVICE)
        self.filter_model.load_state_dict(torch.load("./siren_gru.pth", map_location=DEVICE))
        self.filter_model.eval() 

    def __len__(self):
        return len(self.clean_paths) * 2

    def load_wav(self, path):
        wave, sr = torchaudio.load(path)
        if wave.shape[0] > 1:
            wave = torch.mean(wave, dim=0, keepdim=True)
        if sr != SR:
            wave = torchaudio.functional.resample(wave, sr, SR)
        return wave

    def fix_len(self, wave):
        if wave.shape[1] > self.max_len:
            return wave[:, :self.max_len]
        return torch.nn.functional.pad(wave, (0, self.max_len - wave.shape[1]))

    def __getitem__(self, idx):
                is_siren = torch.rand(1).item() < 0.5

                if is_siren:
                    rand_clean_idx = torch.randint(0, len(self.clean_paths), (1,)).item()
                    clean = self.fix_len(self.load_wav(self.clean_paths[rand_clean_idx]))
                    clean = clean + 1e-3 
                    label = 1.0
                else:
                    clean = torch.full((1, self.max_len), 1e-3) 
                    label = 0.0

                rand_idx = torch.randint(0, len(self.noise_paths), (1,)).item()
                noise = self.fix_len(self.load_wav(self.noise_paths[rand_idx]))
                noise = noise + 1e-3 
                snr = torch.empty(1).uniform_(-5, 5)

                mixed = add_noise(clean, noise, snr)

                with torch.no_grad():
                    m_gt = self.stft(mixed.to(DEVICE))
                    gru_input = m_gt[0].transpose(0, 1).unsqueeze(0)
                    pred_mask = self.filter_model(gru_input)
                    mask_2d = pred_mask.squeeze(0).transpose(0, 1)

                    enhanced_gt = m_gt[0] * mask_2d

                X = enhanced_gt.transpose(0, 1).unsqueeze(0)
                Y = torch.tensor([label], dtype=torch.float32)

                return X.cpu(), Y.cpu()

In [ ]:
class SirenClassifier2D(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        n_frames = SR * DURATION_SEC // HOP_LENGTH + 1
        pooled_time = n_frames // 4
        pooled_bins = N_BINS // 4
        flat_dim = 16 * pooled_time * pooled_bins

        self.fc1 = nn.Linear(flat_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

In [ ]:
if __name__ == "__main__":
    clean_files = glob.glob("/content/drive/MyDrive/dataset/clean_siren/*.wav")
    noise_files = glob.glob("/content/drive/MyDrive/dataset/wind/*.wav")

    if not clean_files or not noise_files:
        raise FileNotFoundError("wav 파일x")

    n_total = len(clean_files)
    n_val = int(n_total * 0.2)
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(42)
    train_clean, val_clean = random_split(clean_files, [n_train, n_val], generator=g)

    train_dataset = SirenClassificationDataset(list(train_clean), noise_files)
    val_dataset = SirenClassificationDataset(list(val_clean), noise_files)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = SirenClassifier2D().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for X_batch, Y_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, Y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        model.eval()
        val_total = 0.0
        with torch.no_grad():
            for X_batch, Y_batch in val_loader:
                X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
                pred = model(X_batch)
                val_total += criterion(pred, Y_batch).item()
        val_loss = val_total / len(val_loader)

        print(f"Epoch {epoch} | train: {train_loss:.5f} | val: {val_loss:.5f}")

    torch.save(model.state_dict(), "siren_classifier.pth")
    print("가중치 파일 저장")